# Audio Synthesis and Sound Design
This notebook covers the four projects in this folder's README:
1. **Synthesizer**: band-limited oscillators, envelopes, and additive, subtractive and FM synthesis (sections 1 to 5).
2. **Physical modeling**: the Karplus-Strong plucked string, and how to tune it exactly (section 6).
3. **Granular synthesis**: rebuilding sound from short grains, including time stretching and pitch shifting (section 7).
4. **Sound effects design**: drum sounds built from simple parts (section 8).

The oscillator, envelope and drum code lives in `audiodsp/synthesis.py`, because the capstones in `09_Capstone_Projects` reuse it. Its source is shown below where it is used.

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, special
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import adsr, drum_loop, hat, kick, polyblep_saw, snare, synthetic_vowels, time_axis

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 44100
rng = np.random.default_rng(0)

def play(x, label, fs=FS):
    print(f'{label}:')
    display.display(display.Audio(np.clip(x, -1, 1), rate=fs, normalize=False))

def show_source(*funcs):
    for func in funcs:
        display.display(display.Code(inspect.getsource(func), language='python'))

def midi_to_hz(note):
    return 440 * 2 ** ((note - 69) / 12)

def level_db_at_bins(y):
    """Spectrum in dB re a full-scale sine of exactly 1 s of audio (1 Hz bins)."""
    return 20 * np.log10(np.abs(np.fft.rfft(y[:FS])) / (FS / 2) + 1e-12)

## 1. Oscillators and aliasing
Classic synthesizer waveforms such as the sawtooth have harmonics that go on forever. Computed naively, sample by sample ($y = 2\,(f t \bmod 1) - 1$), every harmonic above the Nyquist frequency aliases back down as an inharmonic tone. This is the same aliasing as in `00_Audio_Signals`, created inside the computer.

Three ways to make a sawtooth:
- **Naive**: the formula above. Cheap, and audibly aliased at high notes.
- **Additive**: add the harmonics $-\frac{2}{\pi}\sum_k \frac{\sin(2\pi k f t)}{k}$, but only those below the Nyquist frequency. No aliasing at all, but the cost grows with the number of harmonics.
- **PolyBLEP**: the naive waveform, plus a small polynomial correction on the two samples around each jump that rounds it off (a polynomial band-limited step). Almost as cheap as naive, and much cleaner where it matters most.

Aliases near the Nyquist frequency are hard to hear (they are high and weak). Aliases that land low in the spectrum, among the note's own harmonics, are what make naive oscillators sound harsh, so the table measures both.

In [ ]:
show_source(polyblep_saw)

In [ ]:
def naive_saw(f0, n_samples, fs=FS):
    return 2 * ((f0 * np.arange(n_samples) / fs) % 1) - 1

def additive_saw(f0, n_samples, fs=FS):
    t = np.arange(n_samples) / fs
    return -2 / np.pi * sum(np.sin(2 * np.pi * k * f0 * t) / k for k in range(1, int(fs / 2 / f0) + 1))

def worst_alias_db(y, f0, below):
    mag = level_db_at_bins(y)
    freqs = np.arange(len(mag))
    return np.where((freqs % f0 == 0) | (freqs > below), -np.inf, mag).max()

print(f'{"f0":>6}   worst alias below 5 kHz / anywhere (dB)')
print(f'{"":>6}{"naive":>16}{"PolyBLEP":>16}{"additive":>16}')
for f0 in [220, 1000, 3000]:
    row = f'{f0:>4} Hz'
    for make in (naive_saw, lambda f, n: polyblep_saw(f, FS, n), additive_saw):
        y = make(f0, FS)
        row += f'{worst_alias_db(y, f0, 5000):>9.0f} /{worst_alias_db(y, f0, FS / 2):>5.0f}'
    print(row)

sweep_freq = np.geomspace(100, 5000, 4 * FS)
naive_sweep = 2 * (np.cumsum(sweep_freq / FS) % 1) - 1
blep_sweep = polyblep_saw(sweep_freq, FS, len(sweep_freq))
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
for ax, y, name in [(axes[0], naive_sweep, 'naive'), (axes[1], blep_sweep, 'PolyBLEP')]:
    f, t, Sxx = signal.spectrogram(y, FS, nperseg=2048)
    S = 10 * np.log10(Sxx + 1e-15)
    ax.pcolormesh(t, f / 1000, S, shading='auto', vmin=S.max() - 80)
    ax.set_title(f'Sawtooth sweep 100 Hz to 5 kHz, {name}')
    ax.set_xlabel('Time (s)')
axes[0].set_ylabel('Frequency (kHz)')
plt.tight_layout()
plt.show()
play(0.2 * naive_sweep, 'Naive sawtooth sweep')
play(0.2 * blep_sweep, 'PolyBLEP sawtooth sweep')

In the naive sweep, the aliases are the lines that fall while the harmonics rise. Listen for the same falling tones in the naive version.

## 2. Envelopes
A note's loudness changes over time. The classic **ADSR** envelope has four stages: **attack** (rise to full level), **decay** (fall to the sustain level), **sustain** (held while the key is down), and **release** (fade out after the key is released). Applied to amplitude it shapes loudness; applied to a filter's cutoff or an FM index it shapes the tone.

In [ ]:
show_source(adsr)
fig, ax = plt.subplots(figsize=(10, 3))
env = adsr(0.6, FS, attack=0.05, decay=0.15, sustain=0.6, release=0.3)
ax.plot(np.arange(len(env)) / FS, env)
for t_mark, label in [(0.05, 'attack | decay'), (0.2, 'decay | sustain'), (0.6, 'key released')]:
    ax.axvline(t_mark, color='0.6', linestyle=':')
    ax.text(t_mark, 1.02, label, ha='center', fontsize=9)
ax.set_xlabel('Time (s)')
ax.set_ylim(0, 1.15)
plt.show()

## 3. Additive synthesis
Additive synthesis builds a sound from sine waves (**partials**), each with its own frequency, level and envelope.
- An **organ** tone: harmonic partials (integer multiples of the fundamental) with fixed levels, like the drawbars of a Hammond organ, and a simple on/off envelope.
- A **bell**: inharmonic partials (non-integer ratios), each fading at its own rate, the higher ones faster. The inharmonic ratios are what make it sound like struck metal rather than a string or pipe.

In [ ]:
def additive(f0, duration, partials, fs=FS):
    """partials: list of (frequency ratio, level, decay time in seconds or None for no decay)."""
    t = time_axis(duration, fs)
    y = np.zeros_like(t)
    for ratio, level, decay in partials:
        if ratio * f0 < fs / 2:
            envelope = np.exp(-t / decay) if decay else 1.0
            y += level * envelope * np.sin(2 * np.pi * ratio * f0 * t)
    return y

organ_partials = [(1, 1.0, None), (2, 0.6, None), (3, 0.5, None), (4, 0.3, None), (6, 0.2, None), (8, 0.15, None)]
bell_partials = [(0.56, 1.0, 1.5), (0.92, 0.7, 1.2), (1.19, 1.0, 0.9), (1.71, 0.9, 0.7), (2.0, 0.8, 0.6),
                 (2.74, 0.6, 0.45), (3.0, 0.5, 0.35), (3.76, 0.4, 0.25), (4.07, 0.4, 0.2)]

organ = np.concatenate([additive(midi_to_hz(n), 0.45, organ_partials) * adsr(0.4, FS, 0.01, 0.0, 1.0, 0.05)
                        for n in [60, 64, 67, 72]])
bell = additive(440, 3.0, bell_partials)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, y, name in [(axes[0], additive(midi_to_hz(60), 1.0, organ_partials), 'organ, middle C'),
                    (axes[1], bell, 'bell, A = 440 Hz')]:
    segment = y[:FS] * np.hanning(FS)
    mag = 20 * np.log10(np.abs(np.fft.rfft(segment)) + 1e-9)
    ax.plot(np.fft.rfftfreq(FS, 1 / FS), mag - mag.max())
    ax.set_xlim(0, 3000)
    ax.set_ylim(-80, 3)
    ax.set_title(f'Spectrum: {name}')
    ax.set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()
play(0.15 * organ, 'Organ')
play(0.12 * bell, 'Bell')

## 4. Subtractive synthesis
Subtractive synthesis starts from a bright waveform and removes harmonics with a **resonant low-pass filter** whose cutoff moves over time. The filter's **resonance** (its Q) boosts the harmonics near the cutoff, which gives the characteristic "wah" of analog synthesizers as the cutoff sweeps.

The cutoff changes every sample, so the filter needs to stay stable and smooth while its coefficients change. A biquad in direct form does not handle this well. The usual choice is the **state-variable filter** in its topology-preserving form (Andrew Simper's "SVF"), which is a bilinear transform of the analog state-variable filter, pre-warped through $g = \tan(\pi f_c / f_s)$ (`03_Digital_Filters/filter_design_foundations.ipynb`). With a fixed cutoff it gives exactly the same output as the bilinear transform of the analog low-pass $1/(s^2 + s/Q + 1)$, which the first check below confirms.

In [ ]:
def svf_lowpass(x, cutoff, q, fs=FS):
    """Topology-preserving state-variable low-pass (Simper). cutoff: number or per-sample array."""
    g = np.tan(np.pi * np.broadcast_to(np.minimum(cutoff, 0.45 * fs), (len(x),)) / fs)
    k = 1 / q
    a1 = 1 / (1 + g * (g + k))
    a2, a3 = g * a1, g * g * a1
    ic1, ic2 = 0.0, 0.0
    y = np.empty(len(x))
    for n, (v0, b1, b2, b3) in enumerate(zip(x.tolist(), a1.tolist(), a2.tolist(), a3.tolist())):
        v3 = v0 - ic2
        v1 = b1 * ic1 + b2 * v3
        v2 = ic2 + b2 * ic1 + b3 * v3
        ic1, ic2 = 2 * v1 - ic1, 2 * v2 - ic2
        y[n] = v2
    return y

# Check: fixed cutoff equals the pre-warped bilinear transform of the analog prototype
fc, q = 1000.0, 2.0
W = 2 * FS * np.tan(np.pi * fc / FS)
b, a = signal.bilinear([W**2], [1, W / q, W**2], FS)
test = rng.normal(0, 1, FS)
print(f'SVF vs bilinear-transformed analog low-pass: max difference {np.max(np.abs(svf_lowpass(test, fc, q) - signal.lfilter(b, a, test))):.1e}')

In [ ]:
def subtractive_note(freq, duration, cutoff=300, resonance=4.0, env_amount=4000, env_decay=0.15, detune_cents=7):
    n = int((duration + 0.2) * FS)
    detune = 2 ** (detune_cents / 1200)
    oscillators = 0.5 * (polyblep_saw(freq, FS, n) + polyblep_saw(freq * detune, FS, n, phase=0.3))
    t = np.arange(n) / FS
    filter_cutoff = cutoff + env_amount * np.exp(-t / env_decay)        # filter envelope
    amp = adsr(duration, FS, attack=0.005, decay=0.2, sustain=0.6, release=0.2)[:n]
    return svf_lowpass(oscillators, filter_cutoff, resonance) * np.pad(amp, (0, n - len(amp)))

def subtractive_demo(cutoff=300, resonance=4.0, env_amount=4000, env_decay=0.15):
    bassline = [36, 36, 48, 36, 39, 36, 43, 41]
    y = np.concatenate([subtractive_note(midi_to_hz(n), 0.2, cutoff, resonance, env_amount, env_decay)[:int(0.25 * FS)]
                        for n in bassline * 2])
    t = np.arange(int(0.45 * FS)) / FS
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(t, cutoff + env_amount * np.exp(-t / env_decay))
    ax.set_xlabel('Time in note (s)')
    ax.set_ylabel('Cutoff (Hz)')
    ax.set_title('Filter envelope')
    plt.show()
    play(0.4 * y / np.max(np.abs(y)), 'Subtractive bassline')

display.display(interactive(
    subtractive_demo,
    cutoff=widgets.IntSlider(value=300, min=50, max=5000, step=50, description='Cutoff (Hz):'),
    resonance=widgets.FloatSlider(value=4.0, min=0.5, max=20.0, step=0.5, description='Resonance Q:'),
    env_amount=widgets.IntSlider(value=4000, min=0, max=10000, step=100, description='Env (Hz):'),
    env_decay=widgets.FloatSlider(value=0.15, min=0.02, max=1.0, step=0.01, description='Env decay (s):'),
))

## 5. FM synthesis
Frequency modulation (FM) synthesis, made famous by the Yamaha DX7, varies one sine's phase with another sine:

$$y(t) = \sin\!\big(2\pi f_c t + I \sin(2\pi f_m t)\big)$$

$f_c$ is the **carrier** frequency, $f_m$ the **modulator** frequency and $I$ the **modulation index**. The result has sidebands at $f_c \pm n f_m$, with amplitudes given by Bessel functions: $|J_n(I)|$. Two sines can therefore make a rich spectrum. The ratio $f_m / f_c$ sets which frequencies appear (a whole-number ratio gives harmonic tones, others give bell-like inharmonic tones), and the index sets how many sidebands are strong. Making the index follow an envelope makes the brightness change over the note, like real instruments.

The first cell checks the Bessel prediction against a measured spectrum.

In [ ]:
fc, fm, index = 1000, 100, 2.0
t = time_axis(1.0, FS)
y = np.sin(2 * np.pi * fc * t + index * np.sin(2 * np.pi * fm * t))
mag = np.abs(np.fft.rfft(y)) / (FS / 2)
print(f'FM with fc = {fc} Hz, fm = {fm} Hz, I = {index}')
print(f'{"n":>3}{"frequency":>11}{"measured":>10}{"|J_n(I)|":>10}')
for n_side in range(-5, 6):
    print(f'{n_side:>3}{fc + n_side * fm:>8} Hz{mag[fc + n_side * fm]:>10.4f}{abs(special.jv(n_side, index)):>10.4f}')

In [ ]:
def fm_note(fc, duration, ratio=1.0, max_index=5.0, index_decay=0.3, amp_release=0.2):
    t = time_axis(duration + amp_release, FS)
    index = max_index * np.exp(-t / index_decay)
    amp = adsr(duration, FS, attack=0.005, decay=0.1, sustain=0.7, release=amp_release)[:len(t)]
    return amp * np.sin(2 * np.pi * fc * t + index * np.sin(2 * np.pi * ratio * fc * t))

def fm_demo(ratio=1.0, max_index=5.0, index_decay=0.3):
    notes = [60, 64, 67, 72, 67, 64]
    y = np.concatenate([fm_note(midi_to_hz(n), 0.3, ratio, max_index, index_decay)[:int(0.4 * FS)] for n in notes])
    note = fm_note(midi_to_hz(60), 1.0, ratio, max_index, index_decay)
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    for ax, segment, label in [(axes[0], note[:int(0.1 * FS)], 'first 0.1 s'), (axes[1], note[int(0.8 * FS):int(0.9 * FS)], 'at 0.8 s')]:
        f, mag = np.fft.rfftfreq(len(segment), 1 / FS), 20 * np.log10(np.abs(np.fft.rfft(segment * np.hanning(len(segment)))) + 1e-9)
        ax.plot(f, mag - mag.max())
        ax.set_xlim(0, 5000)
        ax.set_ylim(-80, 3)
        ax.set_title(f'Spectrum of middle C, {label}')
        ax.set_xlabel('Frequency (Hz)')
    plt.tight_layout()
    plt.show()
    play(0.3 * y, f'FM, ratio {ratio}, index {max_index}')

display.display(interactive(
    fm_demo,
    ratio=widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.01, description='fm / fc:'),
    max_index=widgets.FloatSlider(value=5.0, min=0.0, max=15.0, step=0.5, description='Index:'),
    index_decay=widgets.FloatSlider(value=0.3, min=0.02, max=3.0, step=0.02, description='Index decay (s):'),
))

Try ratio 1.0 with a fast index decay (brass or plucked), ratio 1.4 with a high index and slow decay (bell), and ratio 3.5 (metallic, inharmonic).

## 6. Physical modeling: the Karplus-Strong string
Instead of describing a sound's spectrum, physical modeling simulates the object that makes it. The **Karplus-Strong** algorithm (1983) models a plucked string with a delay line holding one period of the wave, filled with noise at the pluck. Each pass around the loop, the wave goes through a two-point average, a gentle low-pass filter. High harmonics therefore die out faster than low ones, exactly as on a real string:

$$y[n] = g\,\tfrac{1}{2}\big(y[n-N] + y[n-N-1]\big)$$

The pitch is set by the total loop delay: $N$ samples of delay line plus half a sample from the average. A whole number of samples rarely matches the period $f_s / f_0$, and at high notes the error is large: at 44.1 kHz, C7 has a period of 21.07 samples. The fix (Jaffe and Smith, 1983) is a first-order **all-pass filter** in the loop. It passes every frequency at full level but delays it by the missing fraction $d$ of a sample, with coefficient $C = (1 - d) / (1 + d)$.

The table compares the measured pitch of the untuned string (delay line rounded to whole samples, half-sample delay of the averager ignored) with the tuned one, in **cents** (hundredths of a semitone; trained listeners notice about 5 cents).

In [ ]:
def karplus_strong(f0, duration, decay=0.996, tuned=True, fs=FS, seed=0):
    period = fs / f0
    if tuned:
        N = int(np.floor(period - 0.5 - 0.1))           # keeps the fractional delay d in [0.1, 1.1)
        d = period - 0.5 - N
        C = (1 - d) / (1 + d)
    else:
        N, C = int(round(period)), None
    n_total = int(duration * fs)
    y = np.zeros(n_total)
    y[:N + 1] = np.random.default_rng(seed).uniform(-1, 1, N + 1)   # the pluck: a burst of noise
    y[:N + 1] -= y[:N + 1].mean()
    out = y.tolist()
    v_prev, w_prev = 0.0, 0.0
    for n in range(N + 1, n_total):
        v = decay * 0.5 * (out[n - N] + out[n - N - 1])
        if C is None:
            out[n] = v
        else:
            w = C * v + v_prev - C * w_prev              # all-pass: fractional delay d
            v_prev, w_prev = v, w
            out[n] = w
    return np.array(out)

def measured_pitch(y, f_expected, fs=FS):
    segment = y[int(0.05 * fs):int(0.55 * fs)] * np.hanning(int(0.5 * fs))
    n_fft = 1 << 20
    mag = np.abs(np.fft.rfft(segment, n_fft))
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    near = (freqs > 0.9 * f_expected) & (freqs < 1.1 * f_expected)
    i = np.flatnonzero(near)[np.argmax(mag[near])]
    shift = 0.5 * (mag[i - 1] - mag[i + 1]) / (mag[i - 1] - 2 * mag[i] + mag[i + 1])   # parabolic peak interpolation
    return freqs[i] + shift * fs / n_fft

print(f'{"Note":>5}{"f0":>9}{"untuned error":>16}{"tuned error":>14}')
for name, note in [('A2', 45), ('A3', 57), ('A4', 69), ('E5', 76), ('A5', 81), ('E6', 88), ('C7', 96)]:
    f0 = midi_to_hz(note)
    cents = [1200 * np.log2(measured_pitch(karplus_strong(f0, 0.6, tuned=tuned), f0) / f0) for tuned in (False, True)]
    print(f'{name:>5}{f0:>7.1f} Hz{cents[0]:>12.1f} cents{cents[1]:>9.1f} cents')

melody = [64, 67, 71, 76, 74, 71, 67, 69, 72, 76, 81, 76]
strings = np.zeros(int(4 * FS))
for i, note in enumerate(melody):
    start = int(i * 0.3 * FS)
    tone = karplus_strong(midi_to_hz(note), 1.5, decay=0.997)
    strings[start:start + len(tone)] += tone[:len(strings) - start]
play(0.3 * strings / np.max(np.abs(strings)), 'Karplus-Strong guitar')

## 7. Granular synthesis
Granular synthesis builds sound from **grains**: short snippets, typically 20 to 100 ms, each faded in and out with a smooth window, and overlapped. Where each grain is read from, how fast it is played, and how densely grains are placed are independent controls:
- **Time stretching**: read grains from the source more slowly than they are placed in the output. The sound lasts longer, but each grain still plays at the original speed, so the pitch does not change.
- **Pitch shifting**: play each grain faster or slower (resample it), but place grains at the original pace. The pitch changes, the duration does not.
- **Jitter**: randomize the read positions slightly. This smears the sound into textures and clouds, the classic granular effect.

These are the basic ideas behind the pitch shifter and time stretcher project in `05_Audio_Effects`. Good pitch shifters add analysis to line up grains with the waveform's periods, which avoids the slight roughness heard here.

The source is synthetic speech. Its pitch is measured before and after: the median over all clearly pitched 50 ms frames, each taken from the first strong peak of the normalized autocorrelation (the strongest peak is often at a multiple of the period; `09_Capstone_Projects` section 1 explains why).

In [ ]:
vowels_16k, _ = synthetic_vowels('aiu', sample_rate=16000, f0=150, gap=0.05, lead_in=0.05)
source = signal.resample_poly(vowels_16k, 441, 160)

def granulate(x, stretch=1.0, semitones=0.0, grain_ms=60, overlap=4, jitter_ms=0.0, seed=0):
    grain_rng = np.random.default_rng(seed)
    grain = int(grain_ms / 1000 * FS)
    hop_out = grain // overlap
    rate = 2 ** (semitones / 12)                       # playback speed of each grain
    window = np.hanning(grain)
    n_out = int(len(x) * stretch)
    out, weight = np.zeros(n_out + grain), np.zeros(n_out + grain)
    for out_pos in range(0, n_out, hop_out):
        read_pos = out_pos / stretch + grain_rng.normal(0, jitter_ms / 1000 * FS)
        positions = read_pos + rate * np.arange(grain)
        if positions[0] < 0 or positions[-1] >= len(x) - 1:
            continue
        out[out_pos:out_pos + grain] += window * np.interp(positions, np.arange(len(x)), x)
        weight[out_pos:out_pos + grain] += window
    return (out / np.maximum(weight, 1e-3))[:n_out]

def median_f0(y, lo=60, hi=500):
    """Median pitch over the clearly pitched 50 ms frames of y."""
    frame = int(0.05 * FS)
    lags = np.arange(int(FS / hi), int(FS / lo))
    estimates = []
    for start in range(0, len(y) - frame, frame // 2):
        segment = y[start:start + frame] - y[start:start + frame].mean()
        if np.mean(segment**2) < 1e-6:
            continue                                               # silence
        r = np.correlate(segment, segment, mode='full')[frame - 1:]
        r /= r[0]
        peaks, _ = signal.find_peaks(r[lags])
        if len(peaks) == 0 or r[lags[peaks]].max() < 0.5:
            continue                                               # not clearly pitched
        peaks = lags[peaks]
        estimates.append(FS / peaks[np.argmax(r[peaks] >= 0.9 * r[peaks].max())])
    return np.median(estimates) if estimates else float('nan')

def granular_demo(stretch=2.0, semitones=0.0, grain_ms=60, jitter_ms=0.0):
    y = granulate(source, stretch, semitones, grain_ms, jitter_ms=jitter_ms)
    f_source = median_f0(source)
    print(f'Duration {len(source) / FS:.2f} s -> {len(y) / FS:.2f} s.   '
          f'Median pitch {f_source:.0f} Hz -> {median_f0(y):.0f} Hz (expected {f_source * 2 ** (semitones / 12):.0f} Hz)')
    play(source, 'Source')
    play(y, 'Granulated')

display.display(interactive(
    granular_demo,
    stretch=widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1, description='Stretch:'),
    semitones=widgets.FloatSlider(value=0.0, min=-12, max=12, step=1, description='Pitch (st):'),
    grain_ms=widgets.IntSlider(value=60, min=10, max=200, step=5, description='Grain (ms):'),
    jitter_ms=widgets.FloatSlider(value=0.0, min=0.0, max=100.0, step=5, description='Jitter (ms):'),
))

The measured pitch is a useful check on the method. Pitch shifting lands within about 1% of the target. Time stretching moves the pitch by a few percent, because grains that are not aligned with the waveform's periods interfere with each other. With very short grains (around 10 ms) the grain rate itself becomes an audible pitch, and the measurement reports that instead.

## 8. Sound design: drums
Electronic drum sounds are built from a few simple parts, layered:
- **Kick**: a sine whose pitch drops quickly (150 Hz to 50 Hz in a few tens of milliseconds) gives the "thump", and a short click gives the attack.
- **Snare**: a low tone for the drum shell, plus band-passed noise for the rattling snare wires.
- **Hi-hat**: high-passed noise with a very short decay.

The beat tracker capstone in `09_Capstone_Projects` uses these drums, because a generated beat comes with exact beat times to check against.

In [ ]:
show_source(kick, snare, hat)
for name, sound in [('Kick', kick(FS)), ('Snare', snare(FS, rng)), ('Hi-hat', hat(FS, rng))]:
    play(0.5 * sound / np.max(np.abs(sound)), name)
loop, beat_times = drum_loop(100, 4, FS, rng)
play(loop, 'Drum loop, 100 BPM')